# Preparing Arbitrary Quantum States
## Part II. Alias sampling

As a reminder, this kata consists of two parts:

1. [Part I](./ArbitraryStatePreparation_1_Rotations.ipynb) covers rotation-based state preparation algorithms, starting with naive state preparation.
2. Part II (this notebooks) covers alias sampling, an alternative state preparation algorithm that doesn't use arbitrary rotations.

Using alias sampling algorithm for state preparation was proposed by Craig Gidney; you can read about it in ["Roulette Selection fit for a Quantum Computer"](https://algassert.com/post/1805).

## Classical alias sampling algorithm

The classical alias sampling algorithm solves the roulette selection problem: Given a list of values, pick a random element from the list with probability proportional to its value.

For example, consider the list of values `[7, 1, 3, 5]`. The sum of the values in the list is $16$, and the probabilities of selecting each element in the list should be $\frac7{16}, \frac1{16}, \frac3{16}$, and $\frac5{16}$, respectively.

Alias sampling algorithm reformulates this problem as sampling from a collection of equally likely sub-problems, where each sub-problem involves choosing between only two items. This algorithm turns out to be more efficient (in terms of sampling time) than other roulette selection algorithms. (See the [original blog post](https://algassert.com/post/1805) for comparison.)

Let's illustrate the workings of this algorithm for the example we used above, `[7, 1, 3, 5]`. We can draw the probabilities of picking each value as columns, with each box of each column representing probability $\frac1{16}$. (There are $16$ boxes in the histogram, so the total probability of picking any value is $1$, as expected.)

<center><img src="./images/initial.svg" width=30%/></center>

Next, we rearrange the boxes so that each column is composed of boxes of one or two colors and has height $4$. Here are two examples of how we can do this:

<center>
<table>
  <tr>
    <td style="border: none; padding: 0px; padding-right: 50px;"><img src="./images/repack1.svg" width=100%/></td>
    <td style="border: none; padding: 0px; padding-left: 50px;"><img src="./images/repack2.svg" width=100%/></td>
  </tr>
</table>
</center>

Now, picking a random value looks as follows:

1. Pick a random column (each column has the same probability).
2. Pick a color within that column (the colors and probabilities of choosing each one depend on the choice of the column).
3. Return the value associated with that color.

Before we see how this algorithm can be applied to quantum state preparation, let's see how to implement it as a purely classical algorithm.

### Problem 1. Discretize probabilities

The first step of implementing classical alias sampling algorithm is discretizing the probabilities to represent them as proper fractions with an integer numerator and a denominator that is a power of $2$. (This is not strictly necessary when we're working with floating-point probabilities, but we'll need this once we switch to fixed-point probabilities in quantum state preparation.)

**Inputs:** 

1. A list of $N$ real non-negative numbers `probs` that add up to $1$.
2. An integer $b$ - the number of bits of precision to use for discretization.

**Goal:** 

Return the discretized probabilities `discrete_probs` as a list of $N$ non-negative integers. The normalized probabilities must satisfy two conditions:

1. The sum of all elements in `discrete_probs` must be $2^b$. In other words, dividing each element by $2^b$ must give a valid probability distribution.
2. Each element of `discrete_probs`, divided by $2^b$, must be within $\frac1{2^b}$ absolute tolerance of the corresponding element of `probs`. In other words, the resulting probability distribution must approximate the original probability distribution.

**Examples:**

* `probs = [0.75, 0.25], b = 2` is discretized as `[3, 1]`: $3+1 = 4 = 2^2$, and if you divide each number by $4$, you'll get exactly the original probabilities, no approximation required.
* `probs = [0.3, 0.3, 0.3, 0.1], b = 2` can be discretized as `[1, 1, 1, 1]`: $1 + 1 + 1 + 1 = 4$, and if you divide each number by $4$, you'll get $|0.3 - \frac14| = 0.05 < \frac14$ and $|0.1 - \frac14| = 0.15 < \frac14$. Alternatively, `[2, 1, 1, 0]` is also a valid answer: $|0.3 - \frac24| = 0.2 < \frac14$ and $|0.1 - \frac04| = 0.1 < \frac14$.

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def discretize_probabilities(probs: list[float], b: int) -> list[int]:
    # Write your code here
    ...

### Problem 2. Repack the values

**Inputs:** 

1. An integer $n \ge 1$.
2. An integer $b \ge 0$.
3. A list of $2^n$ non-negative integer values that add up to $2^{n + b}$. Each element of the list, when divided by $2^{n + b}$, is the probability of getting the corresponding outcome during sampling.

**Goal:** "Repack" those values into $2^n$ columns of height $2^b$ described as a list of $2^n$ two-element tuples as follows:

* Each tuple consists of two integers, "alias" and "threshold".
* Tuple with index $i$ describes column $i$ of the repacking. This column consists of two colors: $threshold$ boxes of color $i$ and $2^b - threshold$ boxes of color $alias$. The $alias$ value is allowed to be equal to the column index $i$.

**Example:**

For the example we saw earlier in the kata, the inputs are $n = 2$, $b = 2$, and `values = [7, 3, 1, 5]`. 
Here is how the first repacking of these values is represented as a list of alias-threshold tuples in the format described above:

<center>
<table>
  <tr>
    <td style="border: none; padding: 0px; padding-right: 50px;"><img src="./images/repack1.svg" width=100%/></td>
    <td style="border: none; padding: 0px;">
        <pre>[(0, 4), (0, 3), (3, 1), (0, 2)]</pre>
        or<br/>
        <pre>[(0, 0), (0, 3), (3, 1), (0, 2)]</pre>
    </td>
  </tr>
</table>
</center>

The first column consists of $4$ boxes of the same color. This can be represented either as `threshold = 4` with any value of alias ($4$ boxes of color $0$ and $0$ boxes of color `alias`) or as `alias = 0, threshold = 0` ($0$ boxes of color $0$ and $4$ boxes of color `alias` which is also $0$).

The rest of the columns consist of two colors each, so for them the representation is unique: `threshold` is the number of boxes of the color that matches the column index, and `alias` is the color of the boxes on top of the column (the ones that don't match the column index).

Similarly, the second repacking can be represented as follows:
<center>
<table>
  <tr>
    <td style="border: none; padding: 0px; padding-right: 50px;"><img src="./images/repack2.svg" width=100%/></td>
    <td style="border: none; padding: 0px;">
        <pre>[(0, 4), (3, 3), (0, 1), (0, 4)]</pre>
        or<br/>
        <pre>[(0, 0), (3, 3), (0, 1), (3, 0)]</pre>
    </td>
  </tr>
</table>
</center>

This time, both the first and the last columns have ambiguous representations.

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def repack(n: int, b: int, values: list[int]) -> list[tuple[int, int]]:
    # Write your code here
    ...

### Problem 3. Classical sampling

**Inputs:** 

1. An integer $b \ge 0$.
2. The list of values "repacked" into $2^n$ columns, each of height $2^b$, in the format that matches the output format of the previous problem.

**Goal:** 
Return a single value sampled from the original probability distribution (the one provided as the input to the first problem). In other words, the probability of returning each value $x$ from $0$ to $2^{n} - 1$, inclusive, should be proportional to the number of boxes of color $x$ in the repacked values.

In [ ]:
from test_ArbitraryStatePreparation import problem

@problem
def sample(b: int, repacked_values: list[tuple[int, int]]) -> int:
    # Write your code here
    ...

## Using alias sampling principles for state preparation

$\color{red}{TODO}$

Theory: how to translate classical sampling into quantum state preparation

Choose which column to sample from -> USP to prepare all columns with equal weights
Big second step to figure out which value to choose -> Rearranging the weights between columns

Worked out example or two

n = 1 b = 2 [1, 3] = [2, 6] -> [1/4, 3/4]
(1, 2), (1, 0)

(is [1/8, 7/8] representable?)

XXXX

.  .
.  1
.  1
0  1

.  .
.  .
1  1
0  1


0
0.25
0.5
0.75
1 - not representable

|state_reg> = 1/sqrt2 |0> + 1/sqrt2 |1>
|state_reg>|alias>|threshold> = 1/sqrt2 |0>|1>|1> + 1/sqrt2 |1>|1>|0>
0.5
|state_reg>|alias>|threshold>|coin_toss> = 1/sqrt2 |0>|1>|1>0.5(|0> + |1> + |2> + |3>) + 1/sqrt2 |1>|1>|0>0.5(|0> + |1> + |2> + |3>)

1/sqrt2 * 0.5
|0>|1>|1>|0>
|0>|1>|1>|1>
|1>|0>|1>|2>
|1>|0>|1>|3>

|1>|1>|0>|0>
|1>|1>|0>|1>
|1>|1>|0>|2>
|1>|1>|0>|3>

2|0> + 6|1>

### Problem 4. Load data

**Inputs:** 

1. A `Qubits` register `index` of length $n$ in an arbitrary superposition state.
2. A `Qubits` register `target` of length $m$ in the $\ket{0}$ state.
3. A list of integers `data`, represented as `list[int]`. `data` will have at most $2^n$ elements, each element will be between $0$ and $2^m-1$, inclusive.

**Goal:** Load the data into register `target`, indexed by the value of the register `index`. In other words, for each basis state $\ket{j}$ that is within the range of indices of `data`, transform the input state as follows:

$$\ket{j}_{index}\ket{0}_{target} \rightarrow \ket{j}_{index}\ket{data_j}_{target}$$

For any basis states outside of range of indices of `data`, leave the corresponding components of the input state unchanged.

> Both the index $\ket{j}$ and the value $\ket{data_j}$ should be expressed in binary using little-endian encoding.

In [ ]:
from psiqdk.workbench import Qubits, Qubrick
from test_ArbitraryStatePreparation import problem

@problem
class DataLoader(Qubrick):
    def _compute(self, index: Qubits, target: Qubits, data: list[int]) -> None:
        """Load the data into register `target`, indexed by the value of the register `index`."""
        # Write your code here
        ...

### Problem 5. Swap main register with alias state conditioned on comparison result

**Inputs:** 

1. A `Qubits` register `state_reg` of length $n$ - the register we want to prepare in the superposition state described by the previously given probability distribution.
2. A `Qubits` register `alias` of length $n$ - the register that stores the alias values produced by the repacking algorithm from problem 2.
3. A `Qubits` register `threshold` of length $b$ - the register that stores the threshold values produced by the repacking algorithm from problem 2. These values are stored as fixed-point values between $0$ and $1$ with precision $b$.
4. A `Qubits` register `coin_toss` of length $b$ - the register that stores a fixed-point value between $0$ and $1$ with precision $b$ which we will use to emulate the biased coin toss.

Each register can be in an arbitrary superposition state.

**Goal:** 
Compare the values stored in registers `coin_toss` and `threshold`. If the `coin_toss` value is _greater than or equal to_ `threshold`, swap the states of `state_reg` and `alias`; otherwise, do nothing.

> Feel free to use library Qubricks provided by Workbench for parts of your solution!

In [ ]:
from psiqdk.workbench import Qubits, Qubrick
from test_ArbitraryStatePreparation import problem

@problem
class AliasStateSwap(Qubrick):
    def _compute(self, state_reg: Qubits, alias: Qubits, threshold: Qubits, coin_toss: Qubits) -> None:
        """Swap `state_reg` and `alias` if `coin_toss` >= `threshold`."""
        # Write your code here
        ...

### Problem 6. Complete alias sampling state preparation

$\color{red}{TODO}$

Put it all together
* Classical pre-processing
* Initial USP - library
* Use our data loader (twice)
* Coin toss
* Comparison swap

Test - pull probabilities for register and check that they match the ones we want (same criteria as in task 1)

**Inputs:** 

**Goal:** 

### Demo: Testing alias state preparation routine

$\color{red}{TODO}$

which registers end up entangled with the state register and why it is ok (dig into examples of when this is ok)
Talk (demo) about how to test it using pull probabilities

The following code shows a simple testing harness that you can use to test `AliasSamplingStatePrep` Qubrick from the previous problem.

In [ ]:
from math import atan2, sqrt
from psiqdk.workbench import QPU, Qubits, Qubrick, units
from pytest import approx

class NaiveStatePrep(Qubrick):
    def _compute(self, reg: Qubits, a: list[float], ctrl: Qubits | None = None) -> None:
        """Prepare the register reg in the superposition of all basis states defined by amplitudes a."""
        n = len(reg)
        if n == 1:
            theta = 2 * atan2(a[1], a[0])
            reg.ry(theta * units.rad, cond=ctrl)
        else:
            a_msb0 = a[:2 ** (n - 1)]
            a_msb1 = a[2 ** (n - 1):]
            b0 = sqrt(sum(aj * aj for aj in a_msb0))
            b1 = sqrt(sum(aj * aj for aj in a_msb1))
            
            self._compute(reg[-1], [b0, b1], ctrl)
            self._compute(reg[:-1], a_msb0, ctrl=ctrl | ~reg[-1])
            self._compute(reg[:-1], a_msb1, ctrl=ctrl | reg[-1])


def run_test_state_prep(qubrick: type[Qubrick], num_qubits: int, amps: list[float]) -> None:
    # Create QPU and allocate a fresh register.
    # If your solution uses auxiliary qubits, you'll need to adjust the num_qubits argument accordingly
    qpu = QPU(num_qubits=num_qubits)
    reg = Qubits(num_qubits, 'reg', qpu)
    # Instantiate the state prep Qubrick
    state_prep = qubrick()
    # Prepare the state
    state_prep.compute(reg, amps)
    # Get the state vector
    actual_amps = qpu.pull_state()
    # Check that the actual prepared state matches the expected one
    assert actual_amps == approx(amps)


# Run the test for the Qubrick on a variety of inputs and input sizes
for (n, amps) in [
    (1, [1.0, 0.0]),
    (1, [0.0, -1.0]),
    (1, [-0.6, 0.8]),
    (2, [-1., 0., 0., 0.]),
    (2, [1/sqrt(2), 0., 0., 1/sqrt(2)]),
    (2, [0.36, -0.48, 0.64, -0.48]),
    # More test cases omitted for brevity of the example
]:
    run_test_state_prep(NaiveStatePrep, n, amps)
print("Tests passed!")

### Discussion: When is alias sampling state preparation useful?

$\color{red}{TODO}$

# Part II conclusion

Congratulations! In the second part of this kata you learned to prepare arbitrary quantum states using alias sampling algorithm.
> Copyright (c) 2026 PsiQuantum